In [ ]:
import os
import torch
import pandas as pd
import scanpy as sc
import SpatialGlue
import skmisc
import numpy as np
import torch_geometric
from Model.VGAE import VGAE, train_vgae
from Model.BGRL import BGRL, train_bgrl, get_embedding_bgrl
from Model.DGI import DGI, train_dgi, get_embedding_dgi
from Model.GAE import GAE, train_gae
from Model.GATE import GATE, train_gate
from utils import cKD_refine_label, buildGraph, compute_metrics
from sklearn.mixture import BayesianGaussianMixture

In [2]:
# Environment configuration
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
file_fold = r"dataset"

## Simulated data

In [4]:
# read data 1
rna_df_1 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_100_RNA.csv")
adt_df_1 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_100_ADT.csv")
meta_df_1 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_100_metadata.csv")

In [5]:
# read data 2
rna_df_2 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_1000_RNA.csv")
adt_df_2 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_1000_ADT.csv")
meta_df_2 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_1000_metadata.csv")

In [6]:
# read data 3
rna_df_3 = pd.read_csv(file_fold + r"\Amplify_CBMCs_nCells_5024_nGenes_100_RNA.csv")
adt_df_3 = pd.read_csv(file_fold + r"\Amplify_CBMCs_nCells_5024_nGenes_100_ADT.csv")
meta_df_3 = pd.read_csv(file_fold + r"\Amplify_CBMCs_nCells_5024_nGenes_100_metadata.csv")

In [7]:
# read data 4
rna_df_4 = pd.read_csv(file_fold + r"\Amplify_CBMCs_nCells_5024_nGenes_1000_RNA.csv")
adt_df_4 = pd.read_csv(file_fold + r"\Amplify_CBMCs_nCells_5024_nGenes_1000_ADT.csv")
meta_df_4 = pd.read_csv(file_fold + r"\Amplify_CBMCs_nCells_5024_nGenes_1000_metadata.csv")

In [8]:
# read data 5
rna_df_5 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_100_RNA.csv")
adt_df_5 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_100_ADT.csv")
meta_df_5 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_100_metadata.csv")

In [9]:
# read data 6
rna_df_6 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_1000_RNA.csv")
adt_df_6 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_1000_ADT.csv")
meta_df_6 = pd.read_csv(file_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_1000_metadata.csv")

In [10]:
def create_adata(rna_df, adt_df, meta_df):
    adata_rna = sc.AnnData(X = rna_df.values, obs = meta_df, var = pd.DataFrame(index = rna_df.columns), dtype = "float32")
    adata_adt = sc.AnnData(X = adt_df.values, obs = meta_df, var = pd.DataFrame(index = adt_df.columns), dtype = "float32")

    assert (rna_df.index == meta_df.index).all()
    assert (adt_df.index == meta_df.index).all()
    
    adata_rna.var_names_make_unique()
    adata_adt.var_names_make_unique()
    
    adata_rna.obsm['spatial'] = adata_rna.obs[['X','Y']].to_numpy()
    adata_adt.obsm['spatial'] = adata_adt.obs[['X','Y']].to_numpy()

    return adata_rna, adata_adt

In [11]:
adata_rna_1, adata_adt_1 = create_adata(rna_df = rna_df_1, adt_df = adt_df_1, meta_df = meta_df_1)
adata_rna_2, adata_adt_2 = create_adata(rna_df = rna_df_2, adt_df = adt_df_2, meta_df = meta_df_2)
adata_rna_3, adata_adt_3 = create_adata(rna_df = rna_df_3, adt_df = adt_df_3, meta_df = meta_df_3)
adata_rna_4, adata_adt_4 = create_adata(rna_df = rna_df_4, adt_df = adt_df_4, meta_df = meta_df_4)

c:\Users\hana2\miniconda3\envs\SpatialGlue\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
c:\Users\hana2\miniconda3\envs\SpatialGlue\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
c:\Users\hana2\miniconda3\envs\SpatialGlue\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
c:\Users\hana2\miniconda3\envs\SpatialGlue\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [12]:
adata_rna_5, adata_adt_5 = create_adata(rna_df = rna_df_5, adt_df = adt_df_5, meta_df = meta_df_5)
adata_rna_6, adata_adt_6 = create_adata(rna_df = rna_df_6, adt_df = adt_df_6, meta_df = meta_df_6)

c:\Users\hana2\miniconda3\envs\SpatialGlue\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
c:\Users\hana2\miniconda3\envs\SpatialGlue\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [13]:
from SpatialGlue.preprocess import clr_normalize_each_cell, pca

def preprocess(adata_rna, adata_adt):
    # RNA
    sc.pp.filter_genes(adata_rna, min_cells=10)
    sc.pp.highly_variable_genes(adata_rna, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_rna, target_sum=1e4)
    sc.pp.log1p(adata_rna)
    sc.pp.scale(adata_rna)
    
    adata_rna_high =  adata_rna[:, adata_rna.var['highly_variable']]
    adata_rna.obsm['feat'] = pca(adata_rna_high, n_comps=adata_adt.n_vars-1)

    # Protein
    adata_adt = clr_normalize_each_cell(adata_adt)
    sc.pp.scale(adata_adt)
    adata_adt.obsm['feat'] = pca(adata_adt, n_comps=adata_adt.n_vars-1)

    return adata_rna, adata_adt

In [14]:
adata_rna_1, adata_adt_1 = preprocess(adata_rna_1, adata_adt_1)
adata_rna_2, adata_adt_2 = preprocess(adata_rna_2, adata_adt_2)
adata_rna_3, adata_adt_3 = preprocess(adata_rna_3, adata_adt_3)
adata_rna_4, adata_adt_4 = preprocess(adata_rna_4, adata_adt_4)

In [15]:
adata_rna_5, adata_adt_5 = preprocess(adata_rna_5, adata_adt_5)
adata_rna_6, adata_adt_6 = preprocess(adata_rna_6, adata_adt_6)

In [16]:
# Build graph
data_5 = buildGraph(adata_rna_5, adata_adt_5)
data_6 = buildGraph(adata_rna_6, adata_adt_6)

In [ ]:
epochs = 500

model_5 = VGAE(
    dropout = 0.2,
    in_omics1 = data_5.x_omics1.shape[1],
    in_omics2 = data_5.x_omics2.shape[1],
    recon_omics1_dim = data_5.x_omics1.shape[1],
    recon_omics2_dim = data_5.x_omics2.shape[1],
    recon_spatial_dim = data_5.edge_index.shape[1]
    ).to(device)
z = train_vgae(model_5, data = data_5, epochs = epochs, device = device)

Epoch 001 | Total: 13.8324 | Graph: 2.2805 | KLD: 0.0065 | Omics1: 10.3358 | Omics2: 1.2097
